# Smart Room — Occupancy classification

**Goal:** predict whether the room is occupied (`Occupancy = 1`) from environmental sensor readings only — no cameras.

The notebook follows the order the challenge asks for: dummy baseline → simple model → cross-validated evaluation → one final evaluation on the test set → prediction on new readings.

## Team decisions used here

| Decision | Value |
|---|---|
| Target | `Occupancy` (0 = empty, 1 = occupied) — binary classification |
| Initial features | `Temperature`, `Humidity`, `Light`, `CO2` |
| Left out for now | `date` (only used to order the rows) and `HumidityRatio` (derived from Temperature and Humidity) |
| Train / test split | chronological: the first 80 % of the timeline is the training set, the last 20 % is the test set |
| Cross-validation | `TimeSeriesSplit(n_splits=5)`, inside the training set only |
| Random seed | 42 |
| Metric | F1 of the *occupied* class (precision, recall and accuracy are shown next to it) |
| Missing values | none in this dataset; rows with missing values would be dropped for now |

> **Plugging in the preprocessing pipeline:** only one cell has to change — the one marked `PREPROCESSING INTEGRATION POINT` in section 4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, make_scorer)
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---- shared settings: change them here and nowhere else ----
DATA_PATH = "Occupancy.csv"      # keep the notebook and the CSV in the same folder
TARGET = "Occupancy"
FEATURES = ["Temperature", "Humidity", "Light", "CO2"]
TEST_SIZE = 0.20
N_SPLITS = 5
SEED = 42

pd.set_option("display.precision", 3)

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)   # make the time order explicit

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"From {df['date'].min()}  to  {df['date'].max()}")
df.head()

In [ ]:
print("Missing values per column:")
print(df.isna().sum().to_string())

# Basic missing-value strategy for now: drop incomplete rows (nothing to drop in this dataset)
rows_before = len(df)
df = df.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
print(f"\nRows dropped because of missing values: {rows_before - len(df)}")

print("\nShare of rows per class:")
print(df[TARGET].value_counts(normalize=True).sort_index().rename({0: "empty", 1: "occupied"}).round(3).to_string())

## 2. Chronological train/test split

The readings are taken about once a minute, so neighbouring rows are almost identical. A random split would put the neighbours of every test row into the training set and make the score look better than it really is. It would also let the model learn from the future, which the challenge asks us to avoid.

So we train on the **past** and test on the **future**: the test set is the last 20 % of the timeline and is used only once, at the end.

In [ ]:
n_test = round(len(df) * TEST_SIZE)
split_at = len(df) - n_test

train_df, test_df = df.iloc[:split_at], df.iloc[split_at:]

# X keeps every column except the target (including `date` and `HumidityRatio`),
# so a later preprocessor can use them. The placeholder preprocessor in section 4 picks FEATURES.
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]

pd.DataFrame({
    "rows": [len(train_df), len(test_df)],
    "from": [train_df["date"].min(), test_df["date"].min()],
    "to": [train_df["date"].max(), test_df["date"].max()],
    "share occupied": [y_train.mean(), y_test.mean()],
}, index=["train", "test"])

## 3. How we evaluate

* **Primary metric: F1 of the occupied class.** Only about a quarter of the rows are occupied, so *accuracy* alone is misleading: a model that always answers "empty" is already right most of the time. Precision, recall and accuracy are shown next to F1.
* **Cross-validation** uses `TimeSeriesSplit`: every fold trains on the past and validates on the period right after it. It runs **inside the training set only**.
* **The test set is touched once**, in section 6.

Look at the table below before reading any CV score: the validation folds contain very different shares of occupied rows (some are mostly weekend), so the score of a single fold can swing a lot. Compare the mean ± standard deviation, not one fold.

In [ ]:
cv = TimeSeriesSplit(n_splits=N_SPLITS)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}


def cv_summary(pipeline, name):
    """Cross-validate on the training set only; return mean ± std for every metric."""
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[metric] = f"{values.mean():.3f} ± {values.std():.3f}"
    row["F1 per fold"] = [round(v, 3) for v in scores["test_f1"]]
    return row


# What each validation fold looks like
pd.DataFrame([
    {"fold": i + 1,
     "train rows": len(tr),
     "validation rows": len(va),
     "validation from": X_train["date"].iloc[va].min(),
     "validation to": X_train["date"].iloc[va].max(),
     "share occupied (validation)": y_train.iloc[va].mean()}
    for i, (tr, va) in enumerate(cv.split(X_train))
])

## 4. Preprocessing (placeholder)

The preprocessing track will hand over a full pipeline. Until then this notebook uses the simplest thing that works: keep the four agreed sensor columns and standardise them.

Everything that *learns* from data (here, the scaler's mean and standard deviation) sits **inside the pipeline**, so it is fitted on the training part of each fold only — no leakage.

In [ ]:
# ==================== PREPROCESSING INTEGRATION POINT ====================
# Placeholder: keep the agreed sensor features and standardise them.
# When the preprocessing track hands over its pipeline, replace this cell with:
#
#       from preprocessing import preprocessor
#
# Everything below only needs an unfitted scikit-learn transformer called `preprocessor`
# that accepts the columns of X_train (all columns except the target).
preprocessor = ColumnTransformer(
    [("sensors", StandardScaler(), FEATURES)],
    remainder="drop",        # `date`, `HumidityRatio`... are ignored for now
)
# =========================================================================
preprocessor

## 5. Models and cross-validated results

Four models share the **same** preprocessing, split and metric, so the comparison is fair:

* two dummy baselines — what a "simple prediction" achieves without learning anything from the sensors;
* **Logistic Regression** — the simple model;
* **Random Forest** — a more flexible second model (it does not need scaling, but we keep the same preprocessing for now).

In [ ]:
models = {
    "Dummy (always empty)": DummyClassifier(strategy="most_frequent"),
    "Dummy (random, by class share)": DummyClassifier(strategy="stratified", random_state=SEED),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
}

pipelines = {
    name: Pipeline([("prep", clone(preprocessor)), ("model", model)])
    for name, model in models.items()
}

cv_results = pd.DataFrame([cv_summary(pipe, name) for name, pipe in pipelines.items()]).set_index("model")
cv_results

**How to read this table** (with the default 5 folds)

* A very low F1 in a single fold is a property of the *validation period*, not a bug: the first fold trains on only about two days of data, and folds 2 and 5 are mostly weekend (see the fold table in section 3), where a few false alarms ruin precision. That is why we look at the mean ± standard deviation.
* Both dummies fail by design: "always empty" never finds an occupied minute (F1 = 0), and the random one only reaches an F1 close to the share of occupied rows.
* The test-set scores in section 6 are higher than these CV means because the final models are trained on the whole training set (about 16,000 minutes), while the early folds see far less.

## 6. Final evaluation on the test set (once)

Each pipeline is now fitted on the **whole training set** and evaluated **once** on the test set.

Do not tune or choose models by looking at this table — that would leak the test set into our decisions. It answers one question: *is our model better than a simple prediction?*

In [ ]:
def test_scores(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


fitted, rows = {}, {}
for name, pipe in pipelines.items():
    fitted[name] = pipe.fit(X_train, y_train)
    rows[name] = test_scores(y_test, fitted[name].predict(X_test))

test_results = pd.DataFrame(rows).T
test_results.round(3)

In [ ]:
dummy_names = ["Dummy (always empty)", "Dummy (random, by class share)"]
best_dummy_f1 = test_results.loc[dummy_names, "f1"].max()
always_empty = test_results.loc["Dummy (always empty)"]
lr_f1 = test_results.loc["Logistic Regression", "f1"]

print(f"Always answering 'empty' is right {always_empty['accuracy']:.1%} of the time, but its F1 is {always_empty['f1']:.3f}.")
print(f"Best dummy F1 on the test set:          {best_dummy_f1:.3f}")
print(f"Logistic Regression F1 on the test set: {lr_f1:.3f}")
print("Is our model better than a simple prediction?", "YES" if lr_f1 > best_dummy_f1 else "NO")

In [ ]:
# ---- chart style: light chart surface, one blue hue for magnitude, recessive text ----
SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
BLUE = "#2a78d6"
BLUE_RAMP = LinearSegmentedColormap.from_list(
    "blue_sequential", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])


def ink_on(rgba):
    """White or near-black text, whichever is easier to read on this fill colour."""
    r, g, b = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4 for c in rgba[:3]]
    return "white" if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.18 else INK


def plot_confusion(ax, model, title):
    """Counts in the cells; the colour shows the share of each *actual* class."""
    cm = confusion_matrix(y_test, model.predict(X_test), labels=[0, 1])
    share = cm / cm.sum(axis=1, keepdims=True)
    ax.pcolormesh(share, cmap=BLUE_RAMP, vmin=0, vmax=1, edgecolors=SURFACE, linewidth=3)
    ax.set_xlim(0, 2)
    ax.set_ylim(2, 0)                                   # "Empty" on top
    ax.set_aspect("equal")
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.5, f"{cm[i, j]:,}\n{share[i, j]:.1%}", ha="center", va="center",
                    color=ink_on(BLUE_RAMP(share[i, j])), fontsize=11)
    ax.set_xticks([0.5, 1.5], ["Empty", "Occupied"], color=INK_2)
    ax.set_yticks([0.5, 1.5], ["Empty", "Occupied"], color=INK_2, rotation=90, va="center")
    ax.set_xlabel("Predicted", color=INK_2)
    ax.set_ylabel("Actual", color=INK_2)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=11, fontweight="bold")


fig, axes = plt.subplots(1, 2, figsize=(9, 4.2), facecolor=SURFACE)
for ax, name in zip(axes, ["Logistic Regression", "Random Forest"]):
    plot_confusion(ax, fitted[name], name)
fig.text(0.01, 0.02, "Test set. Each cell: number of minutes and share of that actual class (rows add up to 100 %).",
         color=INK_2, fontsize=9)
fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## 7. Which sensors matter?

For the "explain" part of the demo: which readings does each model rely on? Logistic Regression coefficients are comparable here because the features are standardised. Random Forest importances add up to 100 %.

In [ ]:
def feature_names(pipe):
    """Column names after preprocessing (generic names if the preprocessor cannot provide them)."""
    prep = pipe.named_steps["prep"]
    try:
        return [name.split("__", 1)[-1] for name in prep.get_feature_names_out()]
    except Exception:
        return [f"feature_{i}" for i in range(pipe.named_steps["model"].n_features_in_)]


lr_pipe, rf_pipe = fitted["Logistic Regression"], fitted["Random Forest"]

importance = pd.DataFrame(
    {
        "Logistic Regression: coefficient": lr_pipe.named_steps["model"].coef_[0],
        "Random Forest: importance": rf_pipe.named_steps["model"].feature_importances_,
    },
    index=feature_names(lr_pipe),
).sort_values("Random Forest: importance", ascending=False)
importance.round(3)

In [ ]:
rf_importance = importance["Random Forest: importance"].sort_values()      # largest ends up on top

fig, ax = plt.subplots(figsize=(6.5, 1.0 + 0.55 * len(rf_importance)), facecolor=SURFACE)
bars = ax.barh(rf_importance.index, rf_importance.values, height=0.45, color=BLUE)
for bar, value in zip(bars, rf_importance.values):
    ax.text(value + 0.01, bar.get_y() + bar.get_height() / 2, f"{value:.0%}",
            va="center", ha="left", color=INK, fontsize=10)
ax.set_xlim(0, max(rf_importance.max() * 1.15, 0.1))
ax.set_xticks([])
ax.tick_params(length=0, labelcolor=INK_2, labelsize=10)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_facecolor(SURFACE)
ax.set_title("Random Forest: how much each sensor is used", loc="left", color=INK, fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()

## 8. Prediction on a new reading

`predict_new` takes one sensor reading and returns the answer and the probability of "occupied" (the demo slide shows exactly this). It builds the same columns as the training data, so it keeps working after the preprocessing pipeline is swapped in.

In [ ]:
def humidity_ratio(temperature_c, relative_humidity):
    """Humidity ratio from Temperature (°C) and Humidity (%), at sea-level pressure.
    The dataset's `HumidityRatio` column is derived from the same two readings."""
    saturation_pressure = 610.94 * np.exp(17.625 * temperature_c / (temperature_c + 243.04))
    vapour_pressure = relative_humidity / 100 * saturation_pressure
    return 0.622 * vapour_pressure / (101325 - vapour_pressure)


final_model = fitted["Logistic Regression"]     # the simple model; any other fitted pipeline works too


def predict_new(temperature, humidity, light, co2, when=None, model=final_model, threshold=0.5):
    """Predict whether the room is occupied from ONE new sensor reading."""
    reading = pd.DataFrame([{
        "date": pd.Timestamp(when) if when is not None else pd.Timestamp.now(),
        "Temperature": temperature,
        "Humidity": humidity,
        "Light": light,
        "CO2": co2,
        "HumidityRatio": humidity_ratio(temperature, humidity),
    }])[X_train.columns]                         # same columns, same order as the training data
    probability = float(model.predict_proba(reading)[0, 1])
    return {"occupied": probability >= threshold, "probability_occupied": round(probability, 3)}


# The values shown on the demo slide, and an empty-looking night-time reading
print(predict_new(22.8, 27.4, 410, 850, when="2015-02-18 10:30"))
print(predict_new(20.2, 25.0, 0, 450, when="2015-02-18 02:00"))

## 9. What to do next

1. **Plug in the preprocessing pipeline** (section 4) and re-run the notebook from the top. The split, the metric and the evaluation code do not change.
2. Decide whether `HumidityRatio` and time features (hour, day of week) help. Judge every change with the **cross-validation** table, never with the test set.
3. If an idea needs the previous minutes (rolling averages, the change in CO2), compute it from the **past only**.
4. There is no need to pick a winner yet: this notebook is the shared, reproducible baseline.